In [2]:
from google.colab import files
uploaded=files.upload()

Saving dim_events.csv to dim_events.csv
Saving dim_skus.csv to dim_skus.csv
Saving dim_stores.csv to dim_stores.csv
Saving dim_suppliers.csv to dim_suppliers.csv
Saving fact_inventory_daily.csv to fact_inventory_daily.csv


In [3]:
import pandas as pd
import numpy as np
stores = pd.read_csv("dim_stores.csv")
skus = pd.read_csv("dim_skus.csv")
suppliers = pd.read_csv("dim_suppliers.csv")
events = pd.read_csv("dim_events.csv")
inventory = pd.read_csv("fact_inventory_daily.csv")

In [4]:
#understanding all the 5 datasets and sanity checks
inventory = pd.read_csv("fact_inventory_daily.csv")
print("dim_stores:", stores.shape[0])
print("dim_skus:", skus.shape[0])
print("dim_suppliers:", suppliers.shape[0])
print("dim_events:", events.shape[0])
print("fact_inventory_daily:", inventory.shape[0])


print("STORES COLUMNS:")
print(stores.columns.tolist())
print("\nSKUS COLUMNS:")
print(skus.columns.tolist())
print("\nSUPPLIERS COLUMNS:")
print(suppliers.columns.tolist())
print("\nEVENTS COLUMNS:")
print(events.columns.tolist())
print("\nINVENTORY COLUMNS:")
print(inventory.columns.tolist())


print(inventory['stockout_risk'].value_counts())
print("Start date:", inventory['date'].min())
print("End date:", inventory['date'].max())
print("Unique dates:", inventory['date'].nunique())


print(events[events['event_type'] != 'none'])
festival_check = pd.crosstab(inventory['is_festival_week'],inventory['stockout_risk'],normalize='index')*100
print(festival_check)

suppliers_for_merge = suppliers[['supplier_id','reliability_score']].rename(columns={'reliability_score':'supplier_reliability_score'})
inventory = pd.merge(inventory,suppliers_for_merge,on='supplier_id',how='left')
print(inventory[['supplier_id','supplier_reliability_score']].head())
inventory['reliability_group'] = pd.cut(inventory['supplier_reliability_score'],bins=[-float('inf'),0.75,0.85,float('inf')],labels=['Low','Mid','High'])
print(inventory[['supplier_reliability_score','reliability_group']].head())
reliability_check = pd.crosstab(inventory['reliability_group'],inventory['stockout_risk'],normalize='index')*100
print(reliability_check)

inventory = pd.merge(inventory,skus[['sku_id','is_perishable']],on='sku_id',how='left')
print(inventory[['sku_id','is_perishable']].head())
perishable_check = pd.crosstab(inventory['is_perishable'],inventory['stockout_risk'],normalize='index')*100
print(perishable_check)
print("Missing lead_time_days_actual:",inventory['lead_time_days_actual'].isna().sum())

dim_stores: 12
dim_skus: 60
dim_suppliers: 15
dim_events: 30
fact_inventory_daily: 21600
STORES COLUMNS:
['store_id', 'city', 'city_display', 'city_tier', 'store_size', 'sqft', 'opened_year', 'baseline_daily_orders']

SKUS COLUMNS:
['sku_id', 'sku_name', 'category', 'unit_price_inr', 'shelf_life_days', 'is_perishable', 'festive_relevant', 'popularity_tier', 'supplier_id']

SUPPLIERS COLUMNS:
['supplier_id', 'supplier_name', 'categories_supplied', 'reliability_score', 'base_lead_time_days', 'lead_time_variance_days']

EVENTS COLUMNS:
['date', 'event_name', 'event_type', 'demand_multiplier_festive', 'demand_multiplier_other']

INVENTORY COLUMNS:
['date', 'store_id', 'sku_id', 'supplier_id', 'opening_stock', 'units_demanded', 'units_sold', 'closing_stock', 'reorder_point', 'reorder_placed', 'lead_time_days_expected', 'lead_time_days_actual', 'sales_velocity_7d', 'days_of_cover', 'is_festival_week', 'stockout_risk']
stockout_risk
Safe        14131
At-Risk      5186
Imminent     2283
Name: 

In [5]:
#joining tables

inventory = pd.read_csv("fact_inventory_daily.csv")
inventory = pd.merge(inventory,stores,on='store_id',how='left')
inventory = pd.merge(inventory,skus.drop(columns=['supplier_id']),on='sku_id',how='left')
inventory = pd.merge(inventory,suppliers,on='supplier_id',how='left')
inventory = pd.merge(inventory,events,on='date',how='left')
print(inventory.shape)
print(inventory.columns.tolist())

(21600, 39)
['date', 'store_id', 'sku_id', 'supplier_id', 'opening_stock', 'units_demanded', 'units_sold', 'closing_stock', 'reorder_point', 'reorder_placed', 'lead_time_days_expected', 'lead_time_days_actual', 'sales_velocity_7d', 'days_of_cover', 'is_festival_week', 'stockout_risk', 'city', 'city_display', 'city_tier', 'store_size', 'sqft', 'opened_year', 'baseline_daily_orders', 'sku_name', 'category', 'unit_price_inr', 'shelf_life_days', 'is_perishable', 'festive_relevant', 'popularity_tier', 'supplier_name', 'categories_supplied', 'reliability_score', 'base_lead_time_days', 'lead_time_variance_days', 'event_name', 'event_type', 'demand_multiplier_festive', 'demand_multiplier_other']


In [6]:
#data cleaning
print(inventory.info())
print("Duplicate rows:", inventory.duplicated().sum())
categorical_columns = inventory.select_dtypes(include='object').columns

for column in categorical_columns:
    print("\n", column)
    print(inventory[column].unique())

inventory['city_display'] = inventory['city_display'].str.title()
print(inventory['city_display'].unique())

inventory['date'] = inventory['date'].astype('datetime64[ns]')
print(inventory['date'].dtype)

inventory['supplier_reliability_clean'] = inventory['reliability_score']
inventory['supplier_reliability_clean'] = inventory['supplier_reliability_clean'].fillna(
    inventory.groupby('category')['supplier_reliability_clean'].transform('median')
)
print("Missing reliability values:", inventory['supplier_reliability_clean'].isna().sum())


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21600 entries, 0 to 21599
Data columns (total 39 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   date                       21600 non-null  object 
 1   store_id                   21600 non-null  object 
 2   sku_id                     21600 non-null  object 
 3   supplier_id                21600 non-null  object 
 4   opening_stock              21600 non-null  float64
 5   units_demanded             21600 non-null  int64  
 6   units_sold                 21600 non-null  float64
 7   closing_stock              21600 non-null  float64
 8   reorder_point              21600 non-null  float64
 9   reorder_placed             21600 non-null  object 
 10  lead_time_days_expected    21600 non-null  int64  
 11  lead_time_days_actual      1701 non-null   float64
 12  sales_velocity_7d          21600 non-null  float64
 13  days_of_cover              21600 non-null  flo

In [7]:
inventory = inventory.assign(reorder_gap=inventory['reorder_point'] - inventory['closing_stock'])
print(inventory[['reorder_point', 'closing_stock', 'reorder_gap']].head())

inventory = inventory.assign(
    days_of_cover_ratio=inventory['days_of_cover'] / inventory['lead_time_days_expected']
)
print(inventory[['days_of_cover', 'lead_time_days_expected', 'days_of_cover_ratio']].head())


inventory = inventory.sort_values(['store_id', 'sku_id', 'date'])

inventory['previous_1'] = inventory.groupby(['store_id', 'sku_id'])['reorder_placed'].shift(1)
inventory['previous_2'] = inventory.groupby(['store_id', 'sku_id'])['reorder_placed'].shift(2)
inventory['previous_3'] = inventory.groupby(['store_id', 'sku_id'])['reorder_placed'].shift(3)

inventory['is_recent_reorder'] = (
    (inventory['previous_1'] == 'Y') |
    (inventory['previous_2'] == 'Y') |
    (inventory['previous_3'] == 'Y')
).astype(int)

inventory = inventory.drop(columns=['previous_1', 'previous_2', 'previous_3'])

print(inventory[['store_id', 'sku_id', 'date', 'reorder_placed', 'is_recent_reorder']].head(10))

# Check if 'category' column exists before creating dummy variables
if 'category' in inventory.columns:
    inventory = pd.get_dummies(inventory, columns=['category'], dtype=int)

print(inventory.head())

   reorder_point  closing_stock  reorder_gap
0           36.4          144.9       -108.5
1           36.4          134.9        -98.5
2           36.4          119.9        -83.5
3           36.4          109.9        -73.5
4           36.4          100.9        -64.5
   days_of_cover  lead_time_days_expected  days_of_cover_ratio
0           9.66                        1                 9.66
1          10.79                        1                10.79
2           8.99                        1                 8.99
3           8.79                        1                 8.79
4           8.55                        1                 8.55
  store_id  sku_id       date reorder_placed  is_recent_reorder
0     ST01  SKU001 2026-10-01              N                  0
1     ST01  SKU001 2026-10-02              N                  0
2     ST01  SKU001 2026-10-03              N                  0
3     ST01  SKU001 2026-10-04              N                  0
4     ST01  SKU001 2026-10-05   

In [9]:
inventory = inventory.assign(
    day_of_month=inventory['date'].dt.day
)

print(inventory[['date', 'day_of_month']].head())



festival_start = pd.Timestamp('2026-10-22')

inventory = inventory.assign(
    days_since_festival_start=(inventory['date'] - festival_start).dt.days
)

print(inventory[['date', 'days_since_festival_start']].head())



        date  day_of_month
0 2026-10-01             1
1 2026-10-02             2
2 2026-10-03             3
3 2026-10-04             4
4 2026-10-05             5
        date  days_since_festival_start
0 2026-10-01                        -21
1 2026-10-02                        -20
2 2026-10-03                        -19
3 2026-10-04                        -18
4 2026-10-05                        -17


In [16]:
#model stage 1

print(inventory.columns.tolist())

y = inventory['stockout_risk']

print(y.head())

inventory.select_dtypes(include=np.number)


#enoding useful object columns

categorical_columns = [
    'store_size',
    'is_perishable',
    'festive_relevant',
    'popularity_tier',
    'reorder_placed',
    'is_festival_week',
    'event_type'
]

inventory = pd.get_dummies(
    inventory,
    columns=categorical_columns,
    dtype=int
)

print(inventory.shape)
print(inventory.columns.tolist())




['date', 'store_id', 'sku_id', 'supplier_id', 'opening_stock', 'units_demanded', 'units_sold', 'closing_stock', 'reorder_point', 'lead_time_days_expected', 'lead_time_days_actual', 'sales_velocity_7d', 'days_of_cover', 'stockout_risk', 'city', 'city_display', 'city_tier', 'sqft', 'opened_year', 'baseline_daily_orders', 'sku_name', 'unit_price_inr', 'shelf_life_days', 'supplier_name', 'categories_supplied', 'reliability_score', 'base_lead_time_days', 'lead_time_variance_days', 'event_name', 'demand_multiplier_festive', 'demand_multiplier_other', 'supplier_reliability_clean', 'reorder_gap', 'days_of_cover_ratio', 'is_recent_reorder', 'category_Bakery', 'category_Beverages', 'category_Dairy', 'category_Fruits & Vegetables', 'category_Home Care', 'category_Personal Care', 'category_Snacks', 'category_Staples', 'day_of_month', 'days_since_festival_start', 'store_size_Large', 'store_size_Medium', 'store_size_Small', 'is_perishable_N', 'is_perishable_Y', 'festive_relevant_N', 'festive_relevan

KeyError: "None of [Index(['store_size', 'is_perishable', 'festive_relevant', 'popularity_tier',\n       'reorder_placed', 'is_festival_week', 'event_type'],\n      dtype='object')] are in the [columns]"

In [18]:
#x and y setting

feature_columns = [
    # Raw numerical features
    'opening_stock',
    'units_demanded',
    'units_sold',
    'closing_stock',
    'reorder_point',
    'lead_time_days_expected',
    'sales_velocity_7d',
    'days_of_cover',
    'sqft',
    'opened_year',
    'baseline_daily_orders',
    'unit_price_inr',
    'shelf_life_days',
    'supplier_reliability_clean',
    'base_lead_time_days',
    'lead_time_variance_days',
    'demand_multiplier_festive',
    'demand_multiplier_other',

    # Engineered features
    'supplier_reliability_clean',
    'reorder_gap',
    'days_of_cover_ratio',
    'is_recent_reorder',
    'day_of_month',
    'days_since_festival_start',

    # Category features
    'category_Bakery',
    'category_Beverages',
    'category_Dairy',
    'category_Fruits & Vegetables',
    'category_Home Care',
    'category_Personal Care',
    'category_Snacks',
    'category_Staples',

    # Encoded categorical features
    'store_size_Large',
    'store_size_Medium',
    'store_size_Small',
    'is_perishable_N',
    'is_perishable_Y',
    'festive_relevant_N',
    'festive_relevant_Y',
    'popularity_tier_High',
    'popularity_tier_Low',
    'popularity_tier_Medium',
    'reorder_placed_N',
    'reorder_placed_Y',
    'is_festival_week_N',
    'is_festival_week_Y',
    'event_type_festival',
    'event_type_none',
    'event_type_promo'
]

X = inventory[feature_columns]
y = inventory['stockout_risk']

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Missing values in X:", X.isna().sum().sum())
print("Object columns in X:", X.select_dtypes(include='object').columns.tolist())

X shape: (21600, 49)
y shape: (21600,)
Missing values in X: 0
Object columns in X: []


In [21]:
#train test split

train_data = inventory[inventory['date'] <= '2026-10-23']
test_data = inventory[inventory['date'] >= '2026-10-24']

X_train = train_data[feature_columns]
y_train = train_data['stockout_risk']

X_test = test_data[feature_columns]
y_test = test_data['stockout_risk']

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_test:", X_test.shape)
print("y_test:", y_test.shape)


#base line creation

from sklearn.dummy import DummyClassifier

baseline = DummyClassifier(strategy='most_frequent')

baseline.fit(X_train, y_train)

baseline_predictions = baseline.predict(X_test)

print("Baseline Accuracy:", baseline.score(X_test, y_test))

X_train: (16560, 49)
y_train: (16560,)
X_test: (5040, 49)
y_test: (5040,)
Baseline Accuracy: 0.6228174603174603


In [25]:
#logistic model
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

logistic_model = Pipeline([
    ('scaler', StandardScaler()),
    ('logistic', LogisticRegression(max_iter=1000))
])

logistic_model.fit(X_train, y_train)

logistic_predictions = logistic_model.predict(X_test)

print("Logistic Regression model trained successfully.")

#evaluation logistic
from sklearn.metrics import classification_report

print(classification_report(y_test, logistic_predictions))


#random forest model
from sklearn.ensemble import RandomForestClassifier

random_forest = RandomForestClassifier(
    n_estimators=100,
    criterion='entropy',
    random_state=42
)

random_forest.fit(X_train, y_train)

rf_predictions = random_forest.predict(X_test)

print("Random Forest model trained successfully.")

#random forest evaluation
from sklearn.metrics import classification_report

print(classification_report(y_test, rf_predictions))

Logistic Regression model trained successfully.
              precision    recall  f1-score   support

     At-Risk       0.89      0.82      0.85      1126
    Imminent       0.85      0.85      0.85       775
        Safe       0.97      1.00      0.99      3139

    accuracy                           0.94      5040
   macro avg       0.90      0.89      0.89      5040
weighted avg       0.93      0.94      0.93      5040

Random Forest model trained successfully.
              precision    recall  f1-score   support

     At-Risk       0.83      0.94      0.88      1126
    Imminent       0.89      0.71      0.79       775
        Safe       1.00      1.00      1.00      3139

    accuracy                           0.94      5040
   macro avg       0.90      0.88      0.89      5040
weighted avg       0.94      0.94      0.94      5040



In [26]:
#feature contribution in random forest
feature_importance = pd.DataFrame({
    'Feature': feature_columns,
    'Importance': random_forest.feature_importances_
})

feature_importance = feature_importance.sort_values(
    by='Importance',
    ascending=False
)

print(feature_importance.head(15))

                       Feature  Importance
20         days_of_cover_ratio    0.243350
7                days_of_cover    0.228226
19                 reorder_gap    0.186368
3                closing_stock    0.039299
42            reorder_placed_N    0.023779
6            sales_velocity_7d    0.022493
43            reorder_placed_Y    0.021667
14         base_lead_time_days    0.021631
0                opening_stock    0.020024
5      lead_time_days_expected    0.018244
4                reorder_point    0.017766
21           is_recent_reorder    0.016376
2                   units_sold    0.014759
1               units_demanded    0.014316
18  supplier_reliability_clean    0.011797


In [27]:
#summary
print("Baseline")
print(" Accuracy: 62.28%")
print(" Imminent Recall: 0%")
print()

print("Logistic Regression")
print(" Accuracy: 94%")
print(" Imminent Recall: 85%")
print()

print("Random Forest")
print(" Accuracy: 94%")
print(" Imminent Recall: 71%")

Baseline
 Accuracy: 62.28%
 Imminent Recall: 0%

Logistic Regression
 Accuracy: 94%
 Imminent Recall: 85%

Random Forest
 Accuracy: 94%
 Imminent Recall: 71%
